## Weekly Activity Summary
### Loads intervals.icu activity data from processed CSV files, categorises them based on training focus, and sums weekly totals

In [ ]:
import pandas as pd
import ast
import html

from IPython.display import HTML, display
from hll.config import FIELDS, OUTPUTS_DIR, PROCESSED_DIR, RAW_DIR

pd.set_option("display.max_rows", 200, "display.width", 200)

DAYS = 7
TODAY = pd.Timestamp.today().normalize()
print(f"Today's date: {TODAY}")

View the different fields available for activities

In [ ]:
act = pd.read_csv(PROCESSED_DIR / "activities.csv", parse_dates=["start_date"])
print("activities", act.shape[0])
print("range", act["start_date"].min().date(), "to", act["start_date"].max().date())
act.dtypes.to_frame("dtype").join(act.notna().sum().rename("non-null"))

Group activities by type

In [ ]:
act["date"] = act["start_date"].dt.normalize()

act["mins"] = act["moving_s"] / 60

act["km"] = act["distance"] / 1000

GROUPS = {
    "Run": "run", "VirtualRun": "run", "TrailRun": "run",
    "Ride": "bike", "VirtualRide": "bike",
    "Workout": "hybrid",
    "StairStepper": "cardio", "Elliptical": "cardio", "Rowing": "cardio",
    "WeightTraining": "strength",
}

act["group"] = act["type"].map(GROUPS).fillna("cardio")

act["group"].value_counts()

Create a dataframe which looks specifically at the last 7 days and removes any activities under 5 minutes.

In [ ]:
MIN_MINS = 5 # discount any activities under 5 minutes
start = TODAY - pd.Timedelta(days=DAYS-1) # window includes today

mask = (
    (act["date"] >= start.normalize())
    & (act["date"] <= TODAY.normalize())
    & (act["mins"] >= MIN_MINS)
)

wk = act[mask].copy()

print(f"{start.date()} to {TODAY.date()}: {len(wk)} sessions")


Print weekly summary including number of sessions, hours, distance, training load (from icu_load).

In [ ]:
total_hours = wk["mins"].sum() / 60
total_km = wk["km"].sum()
total_load = wk["icu_load"].sum()

print(f"sessions: {len(wk)}")
print(f"hours: {total_hours:.1f}")
print(f"distance: {total_km:.1f}")
print(f"training load: {total_load:.0f}")


Filter fields by activity type.

In [ ]:
by_group = wk.groupby("group").agg(
    sessions=("mins", "size"),
    hours=("mins", lambda m: m.sum() / 60),
    distance=("km", "sum"),
    load=("icu_load", "sum"),
)

by_group.sort_values("hours", ascending=False).round(1)

Define 'Training Focus' for each activity based on time spent in different heart rate zones.

In [ ]:

def parse_zones(s):
    """Turn the string "[120, 300, ...]" into a real list of numbers."""
    if isinstance(s, str):
        return ast.literal_eval(s)   
    return []                        

wk["zones"] = wk["hr_zone_times"].apply(parse_zones)

wk["z_total"] = wk["zones"].apply(sum)

wk["easy_share"] = wk["zones"].apply(lambda z: sum(z[:2])) / wk["z_total"] # heart rate zone 1/2

wk["hard_share"] = wk["zones"].apply(lambda z: sum(z[3:])) / wk["z_total"] # heart rate zone 4/5
# Sessions with no zone data have z_total = 0, so these become NaN. That's expected.

EASY_SHARE = 0.75     
HARD_SHARE = 0.20    
SHORT_MINS = 20

def training_focus(row):
    """Return a label for one session. Rules are checked in order; first match wins."""
    if row["group"] == "strength":
        return "strength"      # HR is a poor guide for these
    if row["mins"] < SHORT_MINS:
        return "Short / technical"
    if pd.isna(row["easy_share"]):
        return "No HR data"
    if row["hard_share"] >= HARD_SHARE:
        return "High intensity"
    if row["easy_share"] >= EASY_SHARE:
        return "Aerobic base"
    return "Mixed / tempo"

wk["focus"] = wk.apply(training_focus, axis=1)

wk[["group", "mins", "easy_share", "hard_share", "focus"]].round(2)

Build an activity list with a summary of each session.

In [ ]:
cols = ["start_date", "name", "group", "mins", "km", "avg_hr", "load", "focus"]

table = wk[cols].sort_values("start_date").reset_index(drop=True)

table["start_date"] = table["start_date"].dt.strftime("%a %d %b %H:%M")

display(
    table.assign(
        mins=table["mins"].map(lambda x: f"{x:.0f}" if pd.notna(x) else "–"),
        km=table["km"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "–"),
        avg_hr=table["avg_hr"].map(lambda x: f"{x:.0f}" if pd.notna(x) else "–"),
        load=table["load"].map(lambda x: f"{x:.0f}" if pd.notna(x) else "–"),
    )
)

### Create a plot which shows weekly activity totals and individual sessions

In [ ]:
# Colours for the dashboard: dark cards like intervals.icu.
BG, CARD = "#12141f", "#1a1d2e"                       # page background, card background
INK, MUTED, GRID = "#e8eaf2", "#8b90a7", "#2a2e45"    # main text, secondary text, lines

# One colour per group. The coloured edge on each activity row uses these.
COLORS = {
    "run": "#3987e5",
    "bike": "#d95926",
    "hybrid": "#199e70",
    "strength": "#9085e9",
    "cardio": "#c98500",
}

In [ ]:
def tile(label, value, unit):
    """Return the HTML for one stat tile (a small dark card with a big number)."""
    return f"""
    <div style="flex:1; background:{CARD}; border-radius:8px; padding:18px 10px; text-align:center;">
      <div style="color:{MUTED}; font-size:13px;">{label}</div>
      <div style="color:{INK}; font-size:34px; font-weight:700;">{value}</div>
      <div style="color:{MUTED}; font-size:12px;">{unit}</div>
    </div>"""

# Join four tiles into one string. + glues strings together.
tiles_html = (
    tile("This week", len(wk), "sessions")
    + tile("Total time", f"{total_hours:.1f}", "hours")
    + tile("Distance", f"{total_km:.1f}", "km")
    + tile("Training load", f"{total_load:.0f}", "units")
)

# display:flex puts the tiles in one row, and flex:1 on each tile makes them equal width.
tiles_row = f'<div style="display:flex; gap:12px;">{tiles_html}</div>'

In [ ]:
def show(value, decimals=0):
    """Turn a number into text with a set number of decimals; missing values show as a dash."""
    if pd.isna(value):
        return "–"
    return f"{value:.{decimals}f}"        # {decimals} sets the decimal places, e.g. .1f

# Styling for header cells (TH) and data cells (TD).
TH = f"color:{MUTED}; text-align:left; font-weight:500; padding:10px 14px; border-bottom:1px solid {GRID};"
TD = f"padding:10px 14px; border-bottom:1px solid {GRID};"

# Header row: one <th> per column name.
headings = ["Date", "Activity", "Group", "Min", "km", "Avg HR", "Load", "Focus"]
header_html = ""
for h in headings:
    header_html += f'<th style="{TH}">{h}</th>'

# Body: one <tr> per session, newest first.
rows_html = ""
for _, row in wk.sort_values("start_date", ascending=False).iterrows():   # _ is the row number, which we ignore
    edge = COLORS.get(row["group"], MUTED)          # group colour, or grey if the group isn't in COLORS
    name = html.escape(str(row["name"]))            # makes characters like < and & safe inside HTML

    rows_html += (
        "<tr>"
        f'<td style="{TD}">{row["start_date"].strftime("%a %d %b %H:%M")}</td>'
        f'<td style="{TD}">{name}</td>'
        f'<td style="{TD} border-left:4px solid {edge};">{row["group"]}</td>'   # coloured edge marks the group
        f'<td style="{TD}">{show(row["mins"])}</td>'
        f'<td style="{TD}">{show(row["km"], 1)}</td>'
        f'<td style="{TD}">{show(row["avg_hr"])}</td>'
        f'<td style="{TD}">{show(row["icu_load"])}</td>'
        f'<td style="{TD}">{row["focus"]}</td>'
        "</tr>"
    )

table_html = (
    f'<table style="width:100%; border-collapse:collapse; margin-top:12px; '
    f'background:{CARD}; color:{INK}; font-size:14px;">'
    f"<thead><tr>{header_html}</tr></thead><tbody>{rows_html}</tbody></table>"
)

In [ ]:
# One outer box holding a title, the tiles and the activity list.
display(HTML(f"""
<div style="background:{BG}; padding:16px; border-radius:10px; font-family:sans-serif;">
  <div style="color:{INK}; font-size:18px; margin-bottom:12px;">
    Last 7 days · {start:%d %b} – {TODAY:%d %b}
  </div>
  {tiles_row}
  {table_html}
</div>
"""))

## Limitations and notes

### How the numbers are defined
- **Training load** is intervals.icu's own `icu_load`. The data also contains TRIMP, which is not used here.
- **Window:** the last 7 days, including today, based on local start time. Recordings under 5 minutes are dropped.
- **Duration** is moving time, not elapsed time.
- **Distance** is only recorded for some sessions (runs and rides), so the weekly total excludes strength, hybrid and cardio-machine work.
- **Activity groups** come from the intervals.icu activity type. Any type not in the mapping is placed in "cardio", so new types should be checked and added.

### Training focus labels
- Labels are **rule-based heuristics**, not a validated classification. The thresholds (75% of time in Z1–Z2 for "Aerobic base", 20% in Z4+ for "High intensity", under 20 minutes for "Short / technical") are my own choices and can be tuned.
- Zones are grouped into three bands: Z1–Z2 (low), Z3 (moderate), Z4+ (high). The mapping depends on where my intervals.icu zone boundaries sit relative to my thresholds.
- Z3 time is not labelled separately, so tempo-type work can fall into "Mixed / tempo".
- Strength sessions are labelled "strength", as heart rate is not useful

### Heart-rate caveats
- Heart rate lags changes in effort, so short hard intervals tend to under-read in the upper zones.
- Long easy sessions can drift into higher zones, which inflates upper-zone time.
- Heart rate is a weak signal for strength and mixed station work, so load and focus for those sessions are less reliable.

### Scope
- This is a descriptive summary of one athlete over one week. It makes no statistical claims.
- Subjective data (RPE, feel) is mostly missing at this stage, so it is not used here.
- The data is pulled from the intervals.icu API by the sync script and is not stored in the repo, so the notebook needs a recent sync to run.

### Next steps
- Wellness notebook: HRV, resting HR, sleep and a date selector.
- Weekly load, monotony and strain tracker.
- Relating RPE to heart-rate-derived load, so load can be compared when heart rate is unreliable.